# Exercise 02: Text Processing with NLTK
# Cluster 5 - Image, Audio, Text and Database Processing
# Framework: EU AI Act Article 52, UNESCO AI Ethics Recommendation
# Case Study: NCII 🔴

In [ ]:
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
import string
import matplotlib.pyplot as plt

# Download required NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

In [ ]:
# Synthetic social media posts - NCII content moderation scenarios
posts = [
    "I will share your private photos if you don't respond to me",
    "You looked beautiful in those pictures you didn't know I took",
    "I have screenshots of our private conversation and I will post them",
    "Great weather today, really enjoying the sunshine",
    "Just finished reading a great book, highly recommend it",
    "Your career will be over when everyone sees what I have",
    "I know where you live and I have your photos",
    "Had a wonderful time at the park today",
    "You will regret ignoring me, I have ways to make you pay",
    "Looking forward to the weekend"
]

print(f"Total posts: {len(posts)}")
print("\nFirst post:", posts[0])

In [ ]:
# Tokenize each post and analyze
all_tokens = []

for post in posts:
    # Convert to lowercase
    tokens = word_tokenize(post.lower())
    # Remove punctuation
    tokens = [t for t in tokens if t not in string.punctuation]
    all_tokens.extend(tokens)

# Remove stopwords
stop_words = set(stopwords.words('english'))
filtered_tokens = [t for t in all_tokens if t not in stop_words]

print("Total tokens:", len(all_tokens))
print("Tokens after removing stopwords:", len(filtered_tokens))
print("\nFiltered tokens:", filtered_tokens)

In [ ]:
# Most common words in threatening vs normal posts
threatening_posts = posts[:3] + posts[5:7] + [posts[8]]
normal_posts = [posts[3], posts[4], posts[7], posts[9]]

def get_keywords(post_list):
    tokens = []
    for post in post_list:
        t = word_tokenize(post.lower())
        t = [w for w in t if w not in stop_words and w not in string.punctuation]
        tokens.extend(t)
    return FreqDist(tokens)

threat_freq = get_keywords(threatening_posts)
normal_freq = get_keywords(normal_posts)

print("Top threatening words:", threat_freq.most_common(5))
print("Top normal words:", normal_freq.most_common(5))

In [ ]:
# Simple keyword-based classifier
threat_keywords = ['photos', 'private', 'share', 'screenshots', 'career', 
                   'regret', 'pay', 'live', 'post']

def classify_post(post):
    tokens = word_tokenize(post.lower())
    matches = [t for t in tokens if t in threat_keywords]
    if len(matches) >= 2:
        return 'FLAGGED', matches
    return 'CLEAN', []

# Test on all posts
print("Content Moderation Results:")
print("-" * 50)
for i, post in enumerate(posts):
    result, matches = classify_post(post)
    print(f"Post {i+1}: {result} | Matched: {matches}")
    print(f"  Text: {post[:50]}...")

In [ ]:
# Ground truth - which posts are actually threatening?
ground_truth = [1, 1, 1, 0, 0, 1, 1, 0, 1, 0]  # 1=threatening, 0=normal
predictions = []

for post in posts:
    result, _ = classify_post(post)
    predictions.append(1 if result == 'FLAGGED' else 0)

# Calculate metrics
tp = sum(1 for g, p in zip(ground_truth, predictions) if g == 1 and p == 1)
fp = sum(1 for g, p in zip(ground_truth, predictions) if g == 0 and p == 1)
tn = sum(1 for g, p in zip(ground_truth, predictions) if g == 0 and p == 0)
fn = sum(1 for g, p in zip(ground_truth, predictions) if g == 1 and p == 0)

print(f"True Positives (correctly flagged threats): {tp}")
print(f"False Positives (wrongly flagged normal posts): {fp}")
print(f"True Negatives (correctly cleared normal posts): {tn}")
print(f"False Negatives (missed threats): {fn}")
print(f"\nAccuracy: {(tp+tn)/len(posts)*100:.1f}%")
print(f"False Negative Rate (missed threats): {fn/(fn+tp)*100:.1f}%")

## Governance Reflection: NLTK Content Moderation Analysis

A keyword-based moderation system cannot understand context — it matches words, 
not meaning. A clear threat like "your career will be over when everyone sees 
what I have" was classified as CLEAN because it didn't match enough keywords, 
while the actual harm in the sentence was completely invisible to the system.

The false negative rate of 33.3% means 1 in 3 real threats goes undetected. 
For NCII victims, this means the platform takes no action against content that 
is actively harming them. The harm continues while the system reports it as clean.

The EU AI Act Article 52 requires transparency about AI moderation decisions, 
but transparency alone is not enough. The system needs to be context-based, 
not keyword-based. This means requiring platforms to use natural language 
understanding models rather than keyword filters for high-risk content categories 
like NCII — and mandating human review for any content flagged near the threshold.